# Aggregate Existing LCE Store Features - Silver Layer

Aggregates H3 features (from CARTO marketplace) for existing Little Caesars store trade areas.
This provides baseline metrics for comparing expansion candidates.

**Inputs:**
- `{catalog}.{silver_schema}.isochrones_lce` - LCE store trade area polygons
- `{carto_table}` - CARTO Marketplace H3 features (configurable via widget)

**Output:**
- `{catalog}.{silver_schema}.existing_stores_h3` - Aggregated features per existing store

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, explode, lit
from pyspark.sql.window import Window
import yaml

dbutils.widgets.text("catalog", "jdub_demo_aws")
dbutils.widgets.text("silver_schema", "geo_silver")
dbutils.widgets.text("gold_schema", "geo_gold")
dbutils.widgets.text("carto_table", "carto_spatial_features_usa_h3_res_8.carto.derived_spatialfeatures_usa_h3res8_v1_yearly_v3")
dbutils.widgets.text("config_path", "/Workspace/resources/configs/h3_features_config.yml")

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
carto_table = dbutils.widgets.get("carto_table")
config_path = dbutils.widgets.get("config_path")

# Load config
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

H3_RESOLUTION = config['h3_grid']['resolution']

# Table names
isochrones_table = f"{catalog}.{silver_schema}.isochrones_lce"
h3_features_table = carto_table  # Use CARTO table directly
output_table = f"{catalog}.{silver_schema}.existing_stores_h3"

print(f"Input isochrones: {isochrones_table}")
print(f"H3 features (CARTO): {h3_features_table}")
print(f"Output: {output_table}")

In [ ]:
# MAGIC %md
# MAGIC ## Load Trade Areas

In [ ]:
# Load LCE isochrones
trade_areas = spark.table(isochrones_table)

# Standardize columns
columns = trade_areas.columns
id_col = next((c for c in columns if c in ['store_number', 'location_id', 'id']), columns[0])

base_ta = trade_areas.select(
    col(id_col).alias("store_number"),
    col("latitude"),
    col("longitude"),
    col("store_type"),
    col("city") if "city" in columns else lit(None).alias("city"),
    col("state") if "state" in columns else lit(None).alias("state"),
    col("drive_time_minutes") if "drive_time_minutes" in columns else lit(5).alias("drive_time_minutes"),
    col("area_sqkm") if "area_sqkm" in columns else lit(None).alias("area_sqkm"),
    col("geometry")
)

print(f"Loaded {base_ta.count()} existing LCE store trade areas")
display(base_ta.limit(5))

In [ ]:
# MAGIC %md
# MAGIC ## H3 Index Trade Areas

In [ ]:
# Hierarchical H3 indexing (coarse cover → explode → filter)
ta_exploded = base_ta.alias("ta").withColumn(
    "coarse_h3", 
    explode(expr("h3_coverash3string(ST_AsBinary(ta.geometry), 5)"))
).select(
    "ta.*",
    explode(expr(f"h3_tochildren(coarse_h3, {H3_RESOLUTION})")).alias("h3_cell_id")
)

# Filter to cells whose center is within the isochrone
ta_h3 = ta_exploded.alias("exploded").join(
    base_ta.select("store_number", "geometry").alias("orig"),
    expr("orig.store_number = exploded.store_number AND ST_Contains(orig.geometry, ST_GeomFromWKT(h3_centeraswkt(exploded.h3_cell_id), 4326))"),
    "inner"
).select(
    col("exploded.store_number"),
    col("exploded.latitude"),
    col("exploded.longitude"),
    col("exploded.store_type"),
    col("exploded.city"),
    col("exploded.state"),
    col("exploded.drive_time_minutes"),
    col("exploded.area_sqkm"),
    col("exploded.geometry"),
    col("exploded.h3_cell_id")
)

print(f"Trade areas indexed with H3 resolution {H3_RESOLUTION}")
print(f"Total H3 cells across all stores: {ta_h3.count()}")

In [ ]:
# MAGIC %md
# MAGIC ## Join to CARTO H3 Features

In [ ]:
# Load CARTO H3 features
h3_features_raw = spark.table(h3_features_table)

# CARTO uses "h3" as column name, standardize to h3_cell_id
h3_col_name = "h3" if "h3" in h3_features_raw.columns else "h3_cell_id"
h3_features = h3_features_raw.withColumnRenamed(h3_col_name, "h3_cell_id")

# Drop geometry columns if they exist to avoid conflicts
cols_to_drop = ["h3_geometry", "h3_resolution", "processing_timestamp", "geometry"]
for col_name in cols_to_drop:
    if col_name in h3_features.columns:
        h3_features = h3_features.drop(col_name)

# Join trade area H3 cells with CARTO features
ta_with_features = ta_h3.join(h3_features, "h3_cell_id", "inner")

print(f"Joined {ta_with_features.count()} H3 cells with CARTO features")
display(ta_with_features.limit(5))

In [ ]:
# MAGIC %md
# MAGIC ## Aggregate Features by Store

In [ ]:
# Get demographic variables from config
demo_vars = config.get('carto_demographic_variables', {})
count_vars = (
    demo_vars.get('population', []) + 
    demo_vars.get('income', []) + 
    demo_vars.get('households', []) + 
    demo_vars.get('education', []) + 
    demo_vars.get('employment', []) + 
    demo_vars.get('housing', []) + 
    demo_vars.get('commute', [])
)

existing_count_vars = [v for v in count_vars if v in ta_with_features.columns]

# CARTO POI columns
carto_poi_cols = ['retail', 'education', 'financial', 'food_drink', 'healthcare', 'leisure', 'tourism', 'transportation']
existing_poi_cols = [c for c in carto_poi_cols if c in ta_with_features.columns]

print(f"Demographic variables: {len(existing_count_vars)}")
print(f"POI columns: {existing_poi_cols}")

In [ ]:
# Build aggregation expressions
agg_exprs = []

# Sum demographic count variables
for var in existing_count_vars:
    agg_exprs.append(F.abs(F.sum(var)).cast("long").alias(var))

# Sum POI counts
for poi_col in existing_poi_cols:
    agg_exprs.append(F.abs(F.sum(poi_col)).cast("long").alias(f"total_{poi_col}_pois"))

if 'total_poi_count' in ta_with_features.columns:
    agg_exprs.append(F.abs(F.sum("total_poi_count")).cast("long").alias("total_poi_count"))

# Average urbanicity score
if 'urbanicity_score' in ta_with_features.columns:
    agg_exprs.append(F.abs(F.avg("urbanicity_score")).alias("urbanicity_score"))

# Add cell count and geometry
agg_exprs.extend([
    F.count("h3_cell_id").alias("h3_cell_count"),
    F.first("geometry").alias("geometry")
])

# Aggregate by store
ta_features_agg = ta_with_features.groupBy(
    "store_number",
    "latitude",
    "longitude",
    "store_type",
    "city",
    "state",
    "drive_time_minutes",
    "area_sqkm"
).agg(*agg_exprs)

print(f"Aggregated features for {ta_features_agg.count()} stores")
display(ta_features_agg.limit(5))

In [ ]:
# MAGIC %md
# MAGIC ## Write to Silver

In [ ]:
# Add processing timestamp and fill nulls
ta_features_final = ta_features_agg.withColumn("processing_timestamp", F.current_timestamp())

numeric_cols = [
    field.name for field in ta_features_final.schema.fields 
    if field.dataType.typeName() in ['long', 'double', 'integer', 'float']
    and field.name not in ['latitude', 'longitude', 'drive_time_minutes', 'area_sqkm']
]
ta_features_final = ta_features_final.fillna(0, subset=numeric_cols)

# Deduplicate
window_spec = Window.partitionBy("store_number").orderBy(F.desc("processing_timestamp"))
ta_features_final = ta_features_final.withColumn(
    "row_num", F.row_number().over(window_spec)
).filter(F.col("row_num") == 1).drop("row_num")

print(f"Records to write: {ta_features_final.count()}")

# Write to silver
(
    ta_features_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"Written to {output_table}")

In [ ]:
# MAGIC %md
# MAGIC ## Summary Statistics

In [ ]:
print("Existing LCE Store Baseline Metrics:")
display(spark.sql(f"""
  SELECT
    COUNT(*) as total_stores,
    ROUND(AVG(area_sqkm), 2) as avg_trade_area_sqkm,
    ROUND(AVG(population), 0) as avg_population,
    ROUND(AVG(COALESCE(total_retail_pois, 0) + COALESCE(total_food_drink_pois, 0) + COALESCE(total_leisure_pois, 0)), 0) as avg_poi_count,
    ROUND(AVG(h3_cell_count), 0) as avg_h3_cells,
    ROUND(MIN(population), 0) as min_population,
    ROUND(MAX(population), 0) as max_population
  FROM {output_table}
"""))